In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
loaded_model = joblib.load('voting_model_soft')

In [ ]:
import cv2
import numpy as np
from scipy.ndimage import center_of_mass

def predict_my_digit(image_path, model, scaler=None):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return "Không thể đọc được ảnh, vui lòng kiểm tra lại đường dẫn!"

    inverted_img = cv2.bitwise_not(img)
    
    coords = cv2.findNonZero(inverted_img) 
    x, y, w, h = cv2.boundingRect(coords) 
    digit = inverted_img[y:y+h, x:x+w]
    
    if w > h:
        new_w = 20
        new_h = int((20/w)*h)
    else:
        new_h = 20
        new_w = int((20/h)*w)
    resized_digit = cv2.resize(digit, (new_w, new_h))
    
    padded_img = np.zeros((28, 28), dtype=np.uint8)
    start_x = (28 - new_w) // 2
    start_y = (28 - new_h) // 2
    padded_img[start_y:start_y+new_h, start_x:start_x+new_w] = resized_digit
    
    cy, cx = center_of_mass(padded_img)
    shift_x = 14.0 - cx
    shift_y = 14.0 - cy
    
    M = np.float32([[1, 0, shift_x], [0, 1, shift_y]]) 
    shifted_img = cv2.warpAffine(padded_img, M, (28, 28))
    
    flat_img = shifted_img.reshape(1, 784)
    
    if scaler is not None:
        final_img = scaler.transform(flat_img)
    else:
        final_img = flat_img
        
    prediction = model.predict(final_img)
    
    return prediction[0]

In [5]:
for i in range(0,10):
    result = predict_my_digit(f'{i}.png', model=loaded_model, scaler=None)
    print(result)

0
1
2
3
4
5
6
7
8
9
